In [124]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

data = pd.read_csv('../data/used_cars.csv')
data['price'] = data['price'].replace('[\\$,]', '', regex=True).astype(float)
data = data[data['price'] < 130000]
data['milage'] = data['milage'].replace('[^0-9]', '', regex=True).astype(float)
data['horsepower'] = data['engine'].str.extract(r'(\d+\.?\d*)\s*HP', expand=False).astype(float)
data['engine_size'] = data['engine'].str.extract(r'(\d+\.\d+)\s*L(?:iter)?', expand=False).astype(float)
data['age'] = data['model_year'].max() - data['model_year']
data = data.drop(columns=['engine'])
data['horsepower_missing'] = data['horsepower'].isna().astype(int)
data['engine_size_missing'] = data['engine_size'].isna().astype(int)

X = data.drop(['price'], axis=1)
y = data.price


X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=42)
y_train_log = np.log1p(y_train)


In [125]:
num_vals = [c for c in X_train.columns if X_train[c].dtype in ['int64', 'float64']]
cat_vals = X_train.select_dtypes(include=['object', 'string']).columns.tolist()

In [126]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

num_transformer = SimpleImputer(strategy='constant')

cat_transformer = Pipeline(steps=[
    ('Imputer', SimpleImputer(strategy='most_frequent')),
    ('OneHot', OneHotEncoder(handle_unknown='ignore'))
])

my_transformer = ColumnTransformer(transformers=[
    ('num', num_transformer, num_vals),
    ('cat', cat_transformer, cat_vals)
])

In [127]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
import numpy as np

model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=800,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5,
    reg_alpha=0,
    random_state=42
)

final = Pipeline(steps=[
    ('preprocessor', my_transformer),
    ('model', model)
])

# Train
final.fit(X_train, y_train)

# Training prediction
train_log_pred = final.predict(X_train)
#train_pred = np.expm1(train_log_pred)

# Validation prediction
valid_log_pred = final.predict(X_valid)
#valid_pred = np.expm1(valid_log_pred)

# Results
print("Train MAE:", mean_absolute_error(y_train, train_log_pred))
print("Validation MAE:", mean_absolute_error(y_valid, valid_log_pred))

Train MAE: 5517.682758949129
Validation MAE: 6322.569812185879


In [ ]:
print(X_valid.shape)
